## Bronze Layer Ingestion

In this step, we will read the Excel worksheets into PySpark DataFrames and save them as Bronze tables in Databricks.

This will produce **11 tables**:

- **5 operational tables:** one for each operational Excel file.
- **6 reference tables:** one for each worksheet inside `reference_data.xlsx`.

The reference worksheets become separate tables because each contains different information, such as valid statuses, service levels, or approved value mappings.

We will preserve the original dirty values and add source information for traceability. Validation and cleaning will take place in the Silver stage.

## Bronze Operational Data Ingestion

- Install `openpyxl` to read the Excel workbooks.
- Define the catalog, Bronze schema, source volume path, and operational file names.
- List the source files to confirm their location.
- Define a reusable function to read each worksheet and check its column headers.
- Store populated business values as strings and retain blank cells as null to support later validation.
- Create PySpark DataFrames and add the source file path, original Excel row number, and ingestion timestamp.
- Ensure the Bronze schema exists and check for missing files or existing destination tables.
- Save each operational DataFrame as a managed Delta table without overwriting existing tables.
- Compare source and saved row counts to confirm ingestion completeness.
- Preview the saved shipment records.
- Retain duplicates, missing values, and invalid business values for validation in the Silver stage.

In [0]:
#install EXcel reader 
%pip install openpyxl

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Source Configuration

- Set `logistics_quality` as the catalog and `bronze_layer` as the destination schema.
- Define `RAW_PATH` as the volume location containing the raw Excel files.
- Map the five operational table names to their source Excel filenames.
- List the volume contents with `dbutils.fs.ls()` and display them to confirm the files are available.

In [0]:
# let's set the location and the operational files 
CATALOG = "logistics_quality"
SCHEMA = "bronze_layer"
RAW_PATH = "/Volumes/logistics_quality/bronze_layer/raw_data"
operational_files = {
    "shipments": "shipments.xlsx",
    "shipment_events": "shipment_events.xlsx",
    "hubs": "hubs.xlsx",
    "routes": "routes.xlsx",
    "couriers": "couriers.xlsx",
}
display(dbutils.fs.ls(RAW_PATH)) # this function represnt list as a table 

path,name,size,modificationTime
dbfs:/Volumes/logistics_quality/bronze_layer/raw_data/couriers.xlsx,couriers.xlsx,9869,1792612550000
dbfs:/Volumes/logistics_quality/bronze_layer/raw_data/hubs.xlsx,hubs.xlsx,5626,1792612550000
dbfs:/Volumes/logistics_quality/bronze_layer/raw_data/reference_data.xlsx,reference_data.xlsx,13461,1792612550000
dbfs:/Volumes/logistics_quality/bronze_layer/raw_data/routes.xlsx,routes.xlsx,9395,1792612550000
dbfs:/Volumes/logistics_quality/bronze_layer/raw_data/shipment_events.xlsx,shipment_events.xlsx,3389012,1792612576000
dbfs:/Volumes/logistics_quality/bronze_layer/raw_data/shipments.xlsx,shipments.xlsx,986563,1792612561000


### Excel Reading Function

- Import the libraries and capture the ingestion time in UTC.
- Define a function that builds the full destination table name.
- Open each Excel worksheet in read-only mode, retaining formula text.
- Read the column headers and reject empty or duplicate names.
- Convert populated values to strings, keep blanks as null, and record Excel row numbers.
- Create a PySpark DataFrame with source file and ingestion time metadata.
- Return the DataFrame and row count, then close the workbook.

In [0]:
from datetime import datetime,timezone
from pathlib import Path 

from openpyxl import load_workbook 
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)
#capture time when we did run the code 
ingested_at = datetime.now(timezone.utc)
#create function to return the name 
def target_table(table_name):
    return f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"

#function to read the files and set the schema 
def read_operational_excel(file_path,sheet_name):
    workbook = load_workbook(
        file_path,
        read_only = True, #this means no edit
        data_only= False, #this means retain formulas as well
    )
    try:
        worksheet = workbook[sheet_name]
        #read all cell vaues row by row 
        excel_rows = worksheet.iter_rows(values_only=True)
        headers = list(next(excel_rows)) #git first row
        #stop if names are empty 
        if any(not isinstance(name,str)or not name for name in headers):
            raise ValueError(f"Invalid headers in {file_path}")
        if len(headers) != len(set(headers)):
            raise ValueError(f"duplicated headers in {file_path}")
        records = []
        for row_number, row in enumerate(excel_rows, start=2):
            values = [
                None if value is None else str(value)
                for value in row
            ]

            records.append(tuple(values + [row_number]))

        spark_schema = StructType(
            [
                StructField(column, StringType(), True)
                for column in headers
            ]
            + [
                StructField("_source_row_number", LongType(), False)
            ]
        )

        df = (
            spark.createDataFrame(records, schema=spark_schema)
            .withColumn("_source_file", F.lit(file_path))
            .withColumn("_ingested_at", F.lit(ingested_at))
        )

        return df, len(records)

    finally:
        workbook.close()

### Pre-Ingestion Checks

- Create the destination schema if it does not exist.
- Check that every expected operational Excel file exists.
- Stop if a destination table already exists to prevent accidental replacement.
- Confirm readiness to ingest once all checks pass.

In [0]:
#create the destinatio schema inside the inside the catalog if doesn't exists
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`"
)
#loop through the dictionary , the table name is what we want to create and the file name is what we want to read 
for table_name, filename in operational_files.items():
    file_path = f"{RAW_PATH}/{filename}"
    # Check whether a file exists at that path.
    if not Path(file_path).is_file():
        raise FileNotFoundError(file_path)
        
    # Build the full table name using target_table()
    if spark.catalog.tableExists(target_table(table_name)):
        raise ValueError(
            f"{table_name} already exists. "
            "Stop here and review it before running ingestion again."
        )

print("Checks passed. Ready to create the operational tables.")

Checks passed. Ready to create the operational tables.


### Read and Save Operational Data

- Read each operational Excel worksheet into a PySpark DataFrame.
- Save each DataFrame as a Delta table in the destination schema.
- Compare source and saved row counts, stopping if they differ.
- Display a summary of table names and row counts.

In [0]:
#READ and save operational 
#create empty list to collect row-count results
results =[]

for table_name,filename in operational_files.items():
    file_path = f"{RAW_PATH}/{filename}"
    #call the reading function
    df,source_count = read_operational_excel(file_path,sheet_name=table_name,)
    #destination name 
    target = target_table(table_name)
    (
        df.write
        .format("delta")
        .saveAsTable(target)
    )
    saved_count = spark.table(target).count()
    if source_count != saved_count:
        raise RuntimeError(
            f"Row count mismatch for {table_name}:"
            f"{source_count} source rows, {saved_count}")
    results.append((table_name,source_count,saved_count))
#display the results 
display(spark.createDataFrame(
    results,
    ["table_name","source_rows","saved_rows"],
))

table_name,source_rows,saved_rows
shipments,8145,8145
shipment_events,55272,55272
hubs,14,14
routes,136,136
couriers,123,123


### Reference Source Configuration

- Capture the reference ingestion start time in UTC.
- Define the path to `reference_data.xlsx`.
- Map the six reference worksheets to their destination table names.

In [0]:
#capture when reference ingestion starts.
ingested_at = datetime.now(timezone.utc)

#location of reference workbook 
reference_file_path = f"{RAW_PATH}/reference_data.xlsx"
# Map each destination table name to its source worksheet.
reference_sheets = {
"service_levels":"service_levels",
"status_codes":"status_codes",
"exception_reasons":"exception_reasons",
"severity_levels":"severity_levels",
"category_values":"category_values",
"value_aliases":"value_aliases"
}


### Reference Pre-Ingestion Checks

- Confirm that the reference workbook exists.
- Open it in read-only mode and check for the expected worksheets.
- Close the workbook even if a check fails.
- Confirm readiness when all checks pass.

In [0]:
#check if the workbook exists . then check if tables exists on destination 
if not Path(reference_file_path).is_file():
    raise FileNotFoundError(reference_file_path)

#open workbook and inspect it's worksheets names.
workbook = load_workbook(
    reference_file_path,
    read_only = True,
    data_only = False,
)

try:
    for table_name, sheet_name in reference_sheets.items():
        #confirm that the expected worksheet exists.
        if sheet_name not in workbook.sheetnames:
            raise ValueError(
                f"worksheets {sheet_name} is missing from reference_data.xlsx"
                )
            #prevent replacement of an existing table.
            if spark.catalog.tableExists(target_table(table_name)):
                raise ValueError(
                    f"{table_name} already exists"
                    "Review it before running reference ingestion again "
                    )
finally:
    workbook.close()

print("Checks passed. Ready to create the reference worksheets")
    

Checks passed. Ready to create the reference worksheets


### Read and Save Reference Data

- Read each reference worksheet into a PySpark DataFrame.
- Save it as a table in the destination catalog and schema.
- Compare source and saved row counts, stopping if they differ.
- Collect ingestion results and print each created table’s row count.

In [0]:
#read the worksheet and save the delta table 
reference_restults = []
for table_name,sheet_name in reference_sheets.items():
    df,source_count=read_operational_excel(
        file_path = reference_file_path,
        sheet_name = sheet_name
    )
    #destination : cataolg.scheam.table
    target = target_table(table_name)
    #create the delta table 
    (
        df.write
        .saveAsTable(target)

    )
    #count records saved in the destination table 
    saved_count = spark.table(target).count()
    if source_count !=saved_count:
        raise RuntimeError(
            f"Row count mimatch for {table_name}"
            f"{source_count} source rows, {saved_count} saved rows"
            )
    reference_restults.append(
        (table_name,sheet_name,source_count,saved_count)
    )
    print(f"Created {target}:{saved_count}")

Created `logistics_quality`.`bronze_layer`.`service_levels`:3
Created `logistics_quality`.`bronze_layer`.`status_codes`:12
Created `logistics_quality`.`bronze_layer`.`exception_reasons`:7
Created `logistics_quality`.`bronze_layer`.`severity_levels`:4
Created `logistics_quality`.`bronze_layer`.`category_values`:13
Created `logistics_quality`.`bronze_layer`.`value_aliases`:9


### Review Reference Ingestion

- Display table names, source worksheets, and source versus saved row counts.
- Preview the saved `status_codes` table, including its source metadata.

In [0]:
# Show source and saved row counts for all reference tables.
display(
    spark.createDataFrame(
        reference_restults,
        [
            "table_name",
            "source_sheet",
            "source_rows",
            "saved_rows",
        ],
    )
)

# Preview the saved status reference, including its source metadata.
display(
    spark.table(target_table("status_codes"))
)

table_name,source_sheet,source_rows,saved_rows
service_levels,service_levels,3,3
status_codes,status_codes,12,12
exception_reasons,exception_reasons,7,7
severity_levels,severity_levels,4,4
category_values,category_values,13,13
value_aliases,value_aliases,9,9


status_code,terminal_flag,description,_source_row_number,_source_file,_ingested_at
CREATED,False,Created,2,/Volumes/logistics_quality/bronze_layer/raw_data/reference_data.xlsx,2026-10-22T20:45:16.411Z
PICKED_UP,False,Picked up,3,/Volumes/logistics_quality/bronze_layer/raw_data/reference_data.xlsx,2026-10-22T20:45:16.411Z
ORIGIN_RECEIVED,False,Origin received,4,/Volumes/logistics_quality/bronze_layer/raw_data/reference_data.xlsx,2026-10-22T20:45:16.411Z
LINEHAUL_DEPARTED,False,Linehaul departed,5,/Volumes/logistics_quality/bronze_layer/raw_data/reference_data.xlsx,2026-10-22T20:45:16.411Z
DESTINATION_RECEIVED,False,Destination received,6,/Volumes/logistics_quality/bronze_layer/raw_data/reference_data.xlsx,2026-10-22T20:45:16.411Z
OUT_FOR_DELIVERY,False,Out for delivery,7,/Volumes/logistics_quality/bronze_layer/raw_data/reference_data.xlsx,2026-10-22T20:45:16.411Z
DELIVERY_FAILED,False,Delivery failed,8,/Volumes/logistics_quality/bronze_layer/raw_data/reference_data.xlsx,2026-10-22T20:45:16.411Z
DELIVERED,True,Delivered,9,/Volumes/logistics_quality/bronze_layer/raw_data/reference_data.xlsx,2026-10-22T20:45:16.411Z
RETURNED,True,Returned,10,/Volumes/logistics_quality/bronze_layer/raw_data/reference_data.xlsx,2026-10-22T20:45:16.411Z
LOST,True,Lost,11,/Volumes/logistics_quality/bronze_layer/raw_data/reference_data.xlsx,2026-10-22T20:45:16.411Z
